# 🏦 Text-to-SQL financier avec QLoRA

### Fine-tuning efficace d'un petit LLM open source pour interroger des bases financières en langage naturel

![Python](https://img.shields.io/badge/Python-3.11+-3776AB?logo=python&logoColor=white)
![Modèle](https://img.shields.io/badge/Mod%C3%A8le-Qwen2.5--Coder--1.5B-2a78d6)
![Méthode](https://img.shields.io/badge/M%C3%A9thode-QLoRA%20(r%3D16)-eb6834)
![GPU](https://img.shields.io/badge/GPU-T4%20gratuit-76B900?logo=nvidia&logoColor=white)
![Licence](https://img.shields.io/badge/Licence-Apache%202.0-52514e)

|  Résultat |  Gain |  Significativité |  Coût |
|:---:|:---:|:---:|:---:|
| **71,5 %** d'execution accuracy | **+12,7 points** vs baseline (58,9 %) | McNemar **p = 0,0002** | **1,2 %** des paramètres · **18,5 min** · **85 Mo** |

---

**Sommaire**

| | | |
|---|---|---|
| 1. Problématique | 5.  Baseline | 9.  Limitations |
| 2.  État de l'art | 6. 🔧 Entraînement LoRA | 10.  Conclusion |
| 3.  Dataset | 7.  Résultats | 11.  Références |
| 4.  Nettoyage et préparation | 8.  Benchmarking | |

.

##  Installation et configuration

In [ ]:
import os, sys

REPO_URL = "https://github.com/eya-bouhmida/finance-text2sql-lora.git"
REPO_DIR = "/content/finance-text2sql-lora"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull -q
    %cd {REPO_DIR}
    !pip install -q transformers accelerate peft trl bitsandbytes datasets
    !pip uninstall -y -q torchao   # conflit avec peft dans Colab, inutile pour ce projet
    sys.path.append(REPO_DIR)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    sys.path.append(".")

DRIVE_DIR = "/content/drive/MyDrive/text2sql-finance-qlora" if IN_COLAB else "."
DATA_DIR = f"{DRIVE_DIR}/data"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints/qlora-r16"
ADAPTER_DIR = f"{DRIVE_DIR}/adapters/qlora-r16"
RESULTS_DIR = f"{DRIVE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# False : on réutilise l'adaptateur déjà entraîné (sur Drive)
# True  : on relance l'entraînement complet (≈ 20 minutes sur T4)
RELANCER_ENTRAINEMENT = False

In [ ]:
import gc, json, struct, time, warnings
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import data as D, model as M, evaluate as E, train as T, viz as V

warnings.filterwarnings("ignore")
V.setup_style()
pd.set_option("display.max_colwidth", 120)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (active le GPU T4)")

---

## 1.  Problématique

Dans une institution financière, les **analystes, contrôleurs et managers** ont besoin d'interroger des bases SQL, mais ne savent pas écrire de SQL : ils dépendent de l'équipe data, ce qui crée des délais. Les grands LLM propriétaires savent traduire une question en SQL, mais une banque ne peut pas toujours leur envoyer ses schémas et ses questions, pour des raisons de **confidentialité**, de **conformité** et de **coût**.

>  **Question du projet**
> Un petit modèle open source, fine-tuné avec LoRA sur un seul GPU, peut-il générer des requêtes SQL fiables sur des bases financières, tout en restant **déployable en interne** ?

**Fonctionnement du système**

```
 Question en langage naturel ─┐
                                ├─►  LLM fine-tuné ─►  Requête SQL ─► 🗄️Exécution sur la base ─►  Résultat
 Schéma de la base (CREATE) ──┘
```

>  Le modèle ne voit que le **schéma** de la base, **jamais les données** : c'est notre code qui exécute la requête générée.

---

## 2.  État de l'art

| Travaux | Apport | Lien avec ce projet |
|---|---|---|
| **BIRD** (2023) | Benchmark sur bases réelles : les LLM restent loin des humains (≈ 93 %), seuls les très gros modèles atteignent ≈ 80 % | Montre que la tâche reste difficile |
| **BookSQL** (2024) | Text-to-SQL en comptabilité : même GPT-4 échoue souvent | Justifie la spécialisation par domaine |
| **FinSQL** (2024) | Adaptation de LLM à la finance avec des adaptateurs LoRA | Valide l'approche LoRA en finance |

>  Ce projet se place dans un cadre **plus contraint** : un modèle de 1,5 milliard de paramètres et un **seul GPU gratuit**.

---

## 3.  Dataset

**Gretel `synthetic_text_to_sql`** (licence Apache 2.0) : environ 105 000 exemples synthétiques couvrant 100 domaines. Chaque exemple contient une question, le contexte SQL (`CREATE TABLE` + `INSERT`) et la requête de référence. Comme le contexte contient des données, on peut **créer une mini-base SQLite et exécuter les requêtes**.

In [ ]:
train_raw, test_raw = D.load_gretel()
FINANCE_DOMAINS = D.find_finance_domains(train_raw["domain"].unique())
train_fin = D.filter_domains(train_raw, FINANCE_DOMAINS)
test_fin = D.filter_domains(test_raw, FINANCE_DOMAINS)

V.kpi_cards([
    {"label": "Exemples (dataset complet)", "value": f"{len(train_raw) + len(test_raw):,}".replace(",", " "), "accent": V.COLORS["baseline"]},
    {"label": "Domaines", "value": train_raw["domain"].nunique(), "accent": V.COLORS["baseline"]},
    {"label": "Domaines financiers retenus", "value": len(FINANCE_DOMAINS), "sub": ", ".join(FINANCE_DOMAINS)},
    {"label": "Exemples financiers", "value": f"{len(train_fin) + len(test_fin):,}".replace(",", " "),
     "sub": f"{len(train_fin):,} train · {len(test_fin):,} test".replace(",", " ")},
])

In [ ]:
# Un exemple du dataset
cols = ["domain", "sql_complexity", "sql_task_type", "sql_prompt", "sql_context", "sql"]
V.styled(train_fin[cols].head(1).T.rename(columns=lambda _: "Exemple"), caption="Structure d'un exemple")

---

## 4.  Nettoyage et préparation

| Étape | Pourquoi |
|---|---|
|  Lecture seule, requêtes `SELECT` uniquement | Un analyste **interroge** la base, il ne la modifie jamais |
|  Suppression des doublons | Éviter la mémorisation et les fuites entre train et test |
|  Exécution vérifiée dans SQLite | Écarter les requêtes en syntaxe MySQL / PostgreSQL, impossibles à évaluer |
|  Test : résultats vides retirés | Sinon, toute requête qui ne renvoie rien serait comptée juste |
|  Questions du test retirées du train | Le test doit rester inconnu du modèle |

In [ ]:
train_clean, train_report, _ = D.clean_dataset(train_fin)
test_clean, test_report, _ = D.clean_dataset(test_fin, require_non_empty=True)
report = pd.DataFrame({"Train": train_report, "Test": test_report})
V.styled(report, caption="Nombre d'exemples après chaque étape de nettoyage", precision=0).format(
    precision=0, na_rep="—", thousands=" ")

Les fichiers `train.jsonl`, `val.jsonl` et `test.jsonl` issus de cette préparation (séparation 90/10 du train, graine 42) sont stockés sur Drive et réutilisés ci-dessous. Chaque exemple est mis au **format conversation** :

In [ ]:
train_records = D.load_jsonl(f"{DATA_DIR}/train.jsonl")
val_records = D.load_jsonl(f"{DATA_DIR}/val.jsonl")
test_records = D.load_jsonl(f"{DATA_DIR}/test.jsonl")

V.kpi_cards([
    {"label": "Entraînement", "value": len(train_records), "accent": V.COLORS["train"]},
    {"label": "Validation", "value": len(val_records), "accent": V.COLORS["val"]},
    {"label": "Test", "value": len(test_records), "accent": V.COLORS["text"]},
])
V.show_messages(train_records[0]["messages"])

---

## 5.  Baseline

Le modèle de base **Qwen2.5-Coder-1.5B-Instruct** est évalué **sans aucun entraînement** sur les questions de test.

| Métrique | Signification |
|---|---|
| **Execution accuracy**  | La requête générée renvoie le même résultat que la référence (convention BIRD) |
| SQL valide | La requête s'exécute sans erreur |
| Exact match | Le texte SQL est identique à la référence (indicatif) |

>  Décodage **glouton** (déterministe) : les résultats sont reproductibles.

In [ ]:
messages = [r["messages"] for r in test_records]

def run_model(adapter_path=None):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model, tokenizer = M.load_model(M.BASE_MODEL, adapter_path=adapter_path)
    outputs, sec = M.generate(model, tokenizer, messages, batch_size=8)
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    del model
    gc.collect(); torch.cuda.empty_cache()
    return outputs, sec, peak_gb

base_out, base_sec, base_mem = run_model(adapter_path=None)
base_res = E.evaluate_predictions(test_records, base_out)
base_sum = E.summarize(base_res)

V.kpi_cards([
    {"label": "Execution accuracy", "value": f"{V.fr(base_sum['execution_accuracy'])} %", "accent": V.COLORS["baseline"]},
    {"label": "SQL valide", "value": f"{V.fr(base_sum['valid_sql_rate'])} %", "accent": V.COLORS["baseline"]},
    {"label": "Exact match", "value": f"{V.fr(base_sum['exact_match'])} %", "accent": V.COLORS["baseline"]},
])

In [ ]:
V.styled(E.summarize_by(base_res, "sql_complexity"), caption="Baseline par complexité de requête")

>  **À retenir :** le modèle de base maîtrise déjà la syntaxe SQL (plus de 90 % de requêtes valides), mais environ **un tiers de ses réponses sont des requêtes valides qui renvoient un mauvais résultat**. Son point faible est la compréhension de la question et du schéma, plus que la syntaxe.

---

## 6.  Entraînement avec LoRA (QLoRA)

**LoRA** gèle les poids **W** du modèle et apprend, à côté de chaque couche linéaire, une petite correction **ΔW = B·A** de rang *r*. **QLoRA** charge en plus le modèle gelé en **4 bits**, ce qui divise sa mémoire par 4. Seules les matrices A et B sont entraînées.

```
                ┌──── W  (gelé, 4 bits) ─────┐
entrée ───────► │                             ├──► + ──► sortie
                └──── B · A  (entraîné) ─────┘
```

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| `r` / `lora_alpha` | 16 / 32 | Bon compromis capacité / taille ; alpha = 2 × r |
| `target_modules` | toutes les couches linéaires | Recommandation de l'article QLoRA |
| `learning_rate` | 2e-4 | Valeur standard pour LoRA |
| Epochs | 2 | Surapprentissage surveillé par la validation |
| Batch effectif | 16 | 4 exemples × 4 pas d'accumulation |
| Loss | sur la requête SQL uniquement | Le modèle apprend à produire le SQL, pas à recopier le schéma |

In [ ]:
if RELANCER_ENTRAINEMENT:
    train_ds = T.to_prompt_completion(train_records)
    val_ds = T.to_prompt_completion(val_records)
    model, tokenizer = T.load_quantized_model(T.BASE_MODEL)
    trainer = T.build_trainer(model, tokenizer, train_ds, val_ds,
                              T.make_lora_config(), T.make_training_args(CHECKPOINT_DIR))
    start = time.time()
    trainer.train(resume_from_checkpoint=T.latest_checkpoint(CHECKPOINT_DIR))
    train_minutes = round((time.time() - start) / 60, 1)
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    log_history = trainer.state.log_history
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
else:
    # Réutilisation de l'entraînement déjà effectué : historique relu depuis le dernier checkpoint
    with open(f"{T.latest_checkpoint(CHECKPOINT_DIR)}/trainer_state.json") as f:
        log_history = json.load(f)["log_history"]
    train_minutes = 18.5   # mesuré lors de l'entraînement (notebook 03)

# Nombre exact de paramètres entraînés, lu dans l'en-tête du fichier de l'adaptateur
with open(f"{ADAPTER_DIR}/adapter_model.safetensors", "rb") as f:
    header = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
trainable = sum(int(np.prod(v["shape"])) for k, v in header.items() if k != "__metadata__")
TOTAL_BASE_PARAMS = 1.54e9
adapter_mb = T.dir_size_mb(ADAPTER_DIR)

V.kpi_cards([
    {"label": "Paramètres entraînés", "value": f"{V.fr(trainable / 1e6)} M", "sub": "sur 1,54 milliard"},
    {"label": "Part du modèle", "value": f"{V.fr(100 * trainable / TOTAL_BASE_PARAMS)} %"},
    {"label": "Durée d'entraînement", "value": f"{V.fr(train_minutes)} min", "sub": "1 GPU T4 gratuit"},
    {"label": "Taille de l'adaptateur", "value": f"{adapter_mb:.0f} Mo", "sub": "contre ≈ 3 000 Mo pour le modèle"},
])

In [ ]:
logs = pd.DataFrame(log_history)
V.plot_learning_curves(
    logs.dropna(subset=["loss"])[["step", "loss"]],
    logs.dropna(subset=["eval_loss"])[["step", "eval_loss"]],
    path=f"{RESULTS_DIR}/learning_curves.png",
)

> 💡 **À retenir :** pas de sous-apprentissage. Pendant la première epoch, les deux losses baissent ensemble. Au début de la deuxième epoch (≈ étape 180), la loss d'entraînement chute alors que la validation se stabilise : le modèle commence à mémoriser les exemples déjà vus. La validation ne remonte pas, donc **pas de surapprentissage nuisible**, mais une epoch de plus n'apporterait probablement rien. Le checkpoint avec la meilleure loss de validation est conservé.

---

## 7.  Résultats

Le modèle fine-tuné (modèle de base + adaptateur LoRA) est évalué sur **les mêmes questions**, avec **le même chargement** (float16) et **le même décodage** que la baseline.

In [ ]:
ft_out, ft_sec, ft_mem = run_model(adapter_path=ADAPTER_DIR)
ft_res = E.evaluate_predictions(test_records, ft_out)
ft_sum = E.summarize(ft_res)

n = len(test_records)
b, c, p_value = E.mcnemar_test(base_res["execution_match"], ft_res["execution_match"])
gain = ft_sum["execution_accuracy"] - base_sum["execution_accuracy"]

V.kpi_cards([
    {"label": "Baseline", "value": f"{V.fr(base_sum['execution_accuracy'])} %", "sub": "execution accuracy", "accent": V.COLORS["baseline"]},
    {"label": "Fine-tuné QLoRA", "value": f"{V.fr(ft_sum['execution_accuracy'])} %", "sub": "execution accuracy"},
    {"label": "Gain", "value": ("+" if gain >= 0 else "") + f"{V.fr(gain)} pts", "sub": f"{c} corrigées · {b} cassées", "accent": V.COLORS["good"]},
    {"label": "Significativité", "value": f"p = {V.fr(p_value, 4)}", "sub": "test de McNemar exact",
     "accent": V.COLORS["good"] if p_value < 0.05 else V.COLORS["critical"]},
])

In [ ]:
def row(s, res, sec, mem, trained, size):
    lo, hi = E.wilson_ci(int(res["execution_match"].sum()), n)
    return {"Execution accuracy (%)": f"{s['execution_accuracy']:.1f}",
            "Intervalle de confiance 95 %": f"[{lo} ; {hi}]",
            "SQL valide (%)": f"{s['valid_sql_rate']:.1f}",
            "Exact match (%)": f"{s['exact_match']:.1f}",
            "Temps par question (s)": f"{sec:.2f}",
            "Mémoire GPU en inférence (Go)": f"{mem:.1f}",
            "Paramètres entraînés": trained,
            "Taille à stocker": size}

comparison = pd.DataFrame({
    "Baseline": row(base_sum, base_res, base_sec, base_mem, "0", "≈ 3 Go"),
    "Fine-tuné QLoRA": row(ft_sum, ft_res, ft_sec, ft_mem,
                           f"{trainable / 1e6:.1f} M ({100 * trainable / TOTAL_BASE_PARAMS:.1f} %)", f"{adapter_mb:.0f} Mo"),
})
V.styled(comparison, caption="Comparaison complète des deux modèles").set_properties(
    subset=["Fine-tuné QLoRA"], **{"font-weight": "600", "color": V.COLORS["finetuned"]})

---

## 8.  Benchmarking : baseline contre modèle fine-tuné

### 8.1 Le gain est-il significatif ?

Le **test de McNemar** compare les deux modèles question par question, en ne regardant que les questions où ils diffèrent. Une p-value inférieure à 0,05 signifie que l'amélioration **n'est pas due au hasard**.

In [ ]:
both_ok = int((base_res["execution_match"] & ft_res["execution_match"]).sum())
both_ko = int((~base_res["execution_match"] & ~ft_res["execution_match"]).sum())
transitions = pd.DataFrame([[both_ok, b], [c, both_ko]],
                           index=["Baseline ✅ juste", "Baseline ❌ fausse"],
                           columns=["Fine-tuné ✅ juste", "Fine-tuné ❌ faux"])
verdict = "significatif ✅" if p_value < 0.05 else "non significatif ❌"
V.styled(transitions, caption=f"Questions corrigées : {c} · cassées : {b} · p = {p_value:.4f} ({verdict})", precision=0)

### 8.2 Par complexité de requête

In [ ]:
by_cx = pd.DataFrame({
    "n": base_res.groupby("sql_complexity").size(),
    "Baseline (%)": E.summarize_by(base_res)["execution_accuracy"],
    "Fine-tuné (%)": E.summarize_by(ft_res)["execution_accuracy"],
}).sort_values("n", ascending=False)
by_cx["Gain (points)"] = (by_cx["Fine-tuné (%)"] - by_cx["Baseline (%)"]).round(1)

V.plot_by_complexity(by_cx, path=f"{RESULTS_DIR}/accuracy_by_complexity.png")

### 8.3 Types d'erreurs

In [ ]:
def outcome(res):
    return pd.Series({
        "Correct": int(res["execution_match"].sum()),
        "SQL valide mais faux": int((res["executable"] & ~res["execution_match"]).sum()),
        "SQL invalide": int((~res["executable"]).sum()),
    })

outcomes = pd.DataFrame({"Baseline": outcome(base_res), "Fine-tuné": outcome(ft_res)})
V.plot_outcomes(outcomes, path=f"{RESULTS_DIR}/outcomes.png")

>  **À retenir :** le fine-tuning élimine presque toutes les erreurs de syntaxe et augmente nettement la part de réponses correctes. Les requêtes **valides mais fausses** restent la principale source d'erreur : c'est la logique de la requête, plus que la syntaxe, qui reste à améliorer.

### 8.4 Exemples

In [ ]:
merged = base_res[["question", "gold_sql", "pred_sql", "execution_match"]].rename(
    columns={"pred_sql": "baseline_sql", "execution_match": "baseline_ok"})
merged["finetuned_sql"] = ft_res["pred_sql"]
merged["finetuned_ok"] = ft_res["execution_match"]

V.show_examples(merged[~merged["baseline_ok"] & merged["finetuned_ok"]], "✅ Corrigées par le fine-tuning", k=2)
V.show_examples(merged[merged["baseline_ok"] & ~merged["finetuned_ok"]], "❌ Cassées par le fine-tuning", k=2)

In [ ]:
# Sauvegarde des résultats sur Drive
comparison.to_csv(f"{RESULTS_DIR}/comparison.csv")
by_cx.to_csv(f"{RESULTS_DIR}/by_complexity.csv")
merged.to_csv(f"{RESULTS_DIR}/predictions_side_by_side.csv", index=False)
E.save_json({"n_test": n, "baseline": base_sum, "finetuned": ft_sum,
             "mcnemar": {"fixed": c, "broken": b, "p_value": round(p_value, 5)},
             "trainable_params": trainable, "adapter_mb": adapter_mb, "train_minutes": train_minutes},
            f"{RESULTS_DIR}/final_metrics.json")
print("✅ Résultats sauvegardés dans", RESULTS_DIR)

---

## 9.  Limitations

| Limite | Conséquence |
|---|---|
| **Données synthétiques** | Schémas petits (1 à 3 tables) et propres, loin des bases bancaires réelles (centaines de tables, colonnes cryptiques, valeurs sales) |
| **Jeu de test réduit** (158 questions) | Incertitude d'environ ±7 points ; les catégories `window functions`, `subqueries` et `multiple_joins` comptent moins de 10 questions |
| **Même distribution** train / test | Une partie du gain peut refléter l'adaptation au style de Gretel plutôt qu'une meilleure compréhension générale du SQL |
| **Métrique stricte** | Une réponse raisonnable avec une colonne en plus est comptée fausse ; une requête fausse peut tomber juste sur une petite base |
| **Une seule configuration** | Un seul rang (r = 16), une seule graine, pas d'étude d'ablation |
| **Précision** | Adaptateur entraîné en 4 bits puis appliqué en float16 pour l'évaluation (procédure courante) |
| **Périmètre** | SQLite et questions en anglais uniquement |

**🔭 Perspectives :** évaluation sur la base bancaire réelle `financial` du benchmark BIRD · comparaison de plusieurs rangs LoRA (4, 8, 16, 32) · comparaison avec un grand modèle propriétaire · questions en français · intégration dans un agent conversationnel.

---

## 10.  Conclusion

Ce projet montre qu'un petit modèle open source (Qwen2.5-Coder, 1,5 milliard de paramètres), fine-tuné avec QLoRA sur un seul GPU gratuit, améliore nettement la génération de requêtes SQL sur des bases financières : l'execution accuracy passe de **58,9 %** à **71,5 %** (**+12,7 points**) sur 158 questions de test. Ce gain est **statistiquement significatif** (test de McNemar, p = 0,0002) : le fine-tuning corrige 24 questions et n'en casse que 4. Le modèle ne produit presque plus de SQL invalide (de 12 à 2 requêtes), et progresse surtout sur les agrégations (+19 points).

Ce résultat a été obtenu en n'entraînant qu'environ **1,2 % des paramètres**, en **18,5 minutes** sur un GPU T4 gratuit, avec un adaptateur de **85 Mo**, et le modèle reste **déployable en interne**, sans envoyer de données à une API externe. Le principal point faible reste la logique des requêtes : 43 requêtes s'exécutent mais renvoient un mauvais résultat. Ces résultats restent à confirmer sur des bases bancaires réelles, plus complexes que les données synthétiques utilisées ici.

---

## 11.  Références

1. Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
2. Dettmers et al., *QLoRA: Efficient Finetuning of Quantized LLMs*, NeurIPS 2023. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
3. Li et al., *Can LLM Already Serve as a Database Interface? (BIRD)*, NeurIPS 2023. [arXiv:2305.03111](https://arxiv.org/abs/2305.03111)
4. Kumar et al., *BookSQL: A Large Scale Text-to-SQL Dataset for Accounting Domain*, NAACL 2024. [ACL Anthology](https://aclanthology.org/2024.naacl-long.28)
5. Zhang et al., *FinSQL: Model-Agnostic LLMs-based Text-to-SQL Framework for Financial Analysis*, SIGMOD 2024. [arXiv:2401.10506](https://arxiv.org/abs/2401.10506)
6. Gretel AI, *synthetic_text_to_sql* (dataset, Apache 2.0). [Hugging Face](https://huggingface.co/datasets/gretelai/synthetic_text_to_sql)
7. Qwen Team, *Qwen2.5-Coder-1.5B-Instruct* (modèle, Apache 2.0). [Hugging Face](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct)

**Projet réalisé avec l’assistance de Claude (Claude Opus 5.5, Anthropic) pour l’apprentissage des concepts, la structuration du code et le débogage ; les expériences et l’analyse des résultats ont été menées par l’auteure.**